# AI-Based Security Alert Prioritization for SOC Operations

Random Forest classifier that prioritizes SOC alerts (Critical / Medium / Low) using Microsoft's real-world **GUIDE** cybersecurity incident dataset.

Dataset: [Microsoft Security Incident Prediction (GUIDE)](https://www.kaggle.com/datasets/microsoft/microsoft-security-incident-prediction)

In [ ]:
# STEP 1 — Kaggle authentication
# Paste a FRESH token here each session (generate at kaggle.com -> Settings -> API -> Create New Token).
# IMPORTANT: delete the token string from this cell before you save/push the notebook anywhere.
import os
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/access_token', 'w') as f:
    f.write('PASTE_YOUR_KAGGLE_TOKEN_HERE')
os.chmod('/root/.kaggle/access_token', 0o600)

In [ ]:
# STEP 2 — Download the dataset
!pip install -q kaggle
!kaggle datasets download -d microsoft/microsoft-security-incident-prediction
!unzip -q microsoft-security-incident-prediction.zip -d guide_data
!ls guide_data

In [ ]:
# STEP 3 — Load only the columns we need (keeps memory low, avoids the RAM crash)
import pandas as pd

cols_needed = ["IncidentGrade", "Category", "DetectorId", "OSFamily", "SuspicionLevel", "LastVerdict"]
data = pd.read_csv("guide_data/GUIDE_Train.csv", usecols=cols_needed)
print(data.shape)
data.head()

In [ ]:
# STEP 4 — Imports for the rest of the pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

In [ ]:
# STEP 5 — Check for missing values before cleaning
print(data.isnull().sum())

In [ ]:
# STEP 6 — Handle missing values
data["SuspicionLevel"] = data["SuspicionLevel"].fillna("Unknown")
data["LastVerdict"] = data["LastVerdict"].fillna("Unknown")
data["DetectorId"] = data["DetectorId"].fillna(-1)
data["OSFamily"] = data["OSFamily"].fillna(-1)

In [ ]:
# STEP 7 — Map incident grades to priority tiers
priority_map = {
    "TruePositive": "Critical",
    "BenignPositive": "Medium",
    "FalsePositive": "Low"
}

data["Priority"] = data["IncidentGrade"].map(priority_map)
data = data.dropna(subset=["Priority"])  # drop any grade not in the map above
data.head()

In [ ]:
# STEP 8 — Encode categorical columns
le_category = LabelEncoder()
le_suspicion = LabelEncoder()
le_verdict = LabelEncoder()
le_priority = LabelEncoder()

data["Category"] = le_category.fit_transform(data["Category"])
data["SuspicionLevel"] = le_suspicion.fit_transform(data["SuspicionLevel"])
data["LastVerdict"] = le_verdict.fit_transform(data["LastVerdict"])
data["Priority"] = le_priority.fit_transform(data["Priority"])

print("Priority classes:", dict(zip(le_priority.classes_, le_priority.transform(le_priority.classes_))))

In [ ]:
# STEP 9 — Train/test split
X = data[["Category", "SuspicionLevel", "LastVerdict", "DetectorId", "OSFamily"]]
y = data["Priority"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape[0], " Test size:", X_test.shape[0])

In [ ]:
# STEP 10 — Train the model
model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

In [ ]:
# STEP 11 — Evaluate
pred = model.predict(X_test)

acc = accuracy_score(y_test, pred)
print(f"Overall Accuracy: {acc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, pred, target_names=le_priority.classes_))

In [ ]:
# STEP 12 — Confusion matrix
cm = confusion_matrix(y_test, pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le_priority.classes_)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix — Alert Priority Classification")
plt.show()

In [ ]:
# STEP 13 — Feature importance
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print(importance)

importance.plot(x="Feature", y="Importance", kind="bar", legend=False)
plt.title("Factors Influencing Alert Priority")
plt.ylabel("Importance")
plt.tight_layout()
plt.show()

In [ ]:
# STEP 14 — Class distribution
priority_counts = le_priority.inverse_transform(data["Priority"]).astype(str)
counts = pd.Series(priority_counts).value_counts()

print("Total Alerts:", len(data))
print(counts)

In [ ]:
# STEP 15 — Demo prediction on a single new alert
# Safety check first: print valid categories so we pick values that actually exist in this dataset
print("Valid Category values:", list(le_category.classes_)[:15], "...")
print("Valid SuspicionLevel values:", list(le_suspicion.classes_))
print("Valid LastVerdict values:", list(le_verdict.classes_))

In [ ]:
# Pick real values from the lists printed above before running this cell.
# Example below assumes "CredentialAccess" and "Suspicious" exist -- replace if your printout differs.
example_category = "CredentialAccess"
example_suspicion = "Suspicious"
example_verdict = "Suspicious"

new_alert = pd.DataFrame({
    "Category": [le_category.transform([example_category])[0]],
    "SuspicionLevel": [le_suspicion.transform([example_suspicion])[0]],
    "LastVerdict": [le_verdict.transform([example_verdict])[0]],
    "DetectorId": [-1],
    "OSFamily": [-1],
})

prediction = model.predict(new_alert)
print("Predicted Priority:", le_priority.inverse_transform(prediction)[0])

## Results Summary

- **Overall accuracy:** 73.98%
- **Critical-class recall:** 0.75
- **Total alerts analyzed:** 9,465,497 (Medium: 4,110,817 · Critical: 3,322,713 · Low: 2,031,967)
- **Key finding:** `DetectorId` was by far the most influential feature (79.9% importance) — well ahead of `Category` (13.4%), `LastVerdict` (4.7%), `SuspicionLevel` (1.8%), and `OSFamily` (0.1%). This suggests the model leans heavily on which detection rule fired historically, rather than the alert's descriptive content — a reasonable real-world signal, but a limitation worth improving on (e.g. testing model performance with `DetectorId` removed).